<a href="https://colab.research.google.com/github/cdiegor/OtimizacaoCombinatoria/blob/main/VRP_BC_BP_Pyomo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# VRP — Branch-and-Cut (min-cut) e Branch-and-Price (Pyomo, CBC)



In [ ]:
!pip install pyomo networkx gurobipy
!apt-get install -y -qq glpk-utils
!apt-get install -y -qq coinor-cbc



In [ ]:
import math, time, random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from dataclasses import dataclass
from collections import defaultdict, deque
from heapq import heappush, heappop

#opt = pyo.SolverFactory('gurobi', solver_io='python', manage_env=True)
#opt.options['WLSACCESSID'] = '506c9525-d923-4a46-9016-47b019bfde20'
#opt.options['WLSSECRET'] = '6b931547-aef3-4eb4-92c4-005f3f1a158e'
#opt.options['LICENSEID'] = 2750637

# Tenta importar Pyomo
try:
    import pyomo.environ as pyo
except ImportError:
    print("Pyomo não instalado.")

# Configuração Visual
plt.rcParams["figure.figsize"] = (10, 6)

@dataclass
class VRPInstance:
    n: int         # Clientes
    K: int         # Veículos
    Q: int         # Capacidade do Veículo
    coords: np.ndarray
    dist: np.ndarray

def make_instance(n=20, K=4, Q=6, seed=42):
    np.random.seed(seed)
    depot = np.array([[50.0, 50.0]]) # Centro
    # Clientes espalhados
    clients = np.random.uniform(0, 100, size=(n, 2))
    coords = np.vstack([depot, clients])

    diff = coords[:,None,:] - coords[None,:,:]
    dist = np.sqrt(np.sum(diff**2, axis=2))

    return VRPInstance(n=n, K=K, Q=Q, coords=coords, dist=dist)

def plot_routes(inst, routes, title="Rotas"):
    plt.figure()
    # Plota arestas das rotas
    colors = plt.cm.tab10(np.linspace(0, 1, len(routes)))

    for idx, r in enumerate(routes):
        r_coords = inst.coords[r]
        # Adiciona o retorno ao zero se não tiver
        if r[-1] != 0:
            r_coords = np.vstack([r_coords, inst.coords[0]])

        plt.plot(r_coords[:,0], r_coords[:,1], c=colors[idx], linewidth=2, label=f"V{idx+1}", zorder=1)
        # Setas
        mid_idx = len(r)//2
        plt.arrow(r_coords[mid_idx,0], r_coords[mid_idx,1],
                  (r_coords[mid_idx+1,0]-r_coords[mid_idx,0])*0.01,
                  (r_coords[mid_idx+1,1]-r_coords[mid_idx,1])*0.01,
                  color=colors[idx], head_width=2)

    # Plota nós
    plt.scatter(inst.coords[1:,0], inst.coords[1:,1], c='steelblue', s=100, zorder=2)
    plt.scatter(inst.coords[0,0], inst.coords[0,1], c='orange', marker='s', s=150, zorder=3, label="Depósito")

    for i in range(1, inst.n+1):
        plt.text(inst.coords[i,0]+1, inst.coords[i,1]+1, str(i), fontsize=9, fontweight='bold')

    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def get_solver(max_seconds=300):
    # Prioriza Gurobi, CBC, depois GLPK

    #opt = pyo.SolverFactory('gurobi', solver_io='python', manage_env=True)
    #if opt.available():
    #    opt.options['TimeLimit'] = max_seconds
    #    return 'gurobi', opt
    opt = pyo.SolverFactory('cbc')
    if opt.available():
        opt.options['seconds'] = max_seconds
        return 'cbc', opt
    opt = pyo.SolverFactory('glpk')
    if opt.available():
        opt.options['tmlim'] = max_seconds
        return 'glpk', opt
    return None, None



In [ ]:
# --- CRIAÇÃO DA INSTÂNCIA ---
inst = make_instance(n=30, K=4, Q=30, seed=42)
plot_routes(inst, [], "Instância VRP ")

## Instâncias e visualização


## Parte I — Branch-and-Cut com **out-cuts** via **min-cut**

Modelo base (m-TSP por arcos; graus em clientes = 1; $K$ saídas/entradas no depósito).  
Os **out-cuts** $\sum_{i\in S}\sum_{j\notin S} x_{ij} \ge 1$ são separados resolvendo **min-cut** $(0\to t)$ para cada cliente $t$.  
Agora com **deduplicação de cortes**, **tolerâncias** e opção de **laço em LP**.


Modelo Matemático: Branch-and-Cut (2-Index Flow)Seja $G = (V, A)$ um grafo onde $V = \{0, ..., n\}$ é o conjunto de nós (0 é o depósito) e $A$ o conjunto de arcos.

Variáveis de Decisão:$$x_{ij} = \begin{cases} 1 & \text{se o veículo viaja do nó } i \text{ para } j \\ 0 & \text{caso contrário} \end{cases}$$

Função Objetivo:Minimizar o custo total de transporte:

$$\min \sum_{i \in V} \sum_{j \in V, i \neq j} c_{ij} x_{ij}$$

Restrições:

Grau de Entrada e Saída: Cada cliente deve ser visitado exatamente uma vez.$$\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V \setminus \{0\}$$

$$\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V \setminus \{0\}$$

Fluxo no Depósito: O número de veículos saindo e voltando ao depósito deve ser igual a $K$ (número de veículos).

$$\sum_{j \in V \setminus \{0\}} x_{0j} = K$$

$$\sum_{i \in V \setminus \{0\}} x_{i0} = K$$

Eliminação de Subciclos (Capacity Cuts): Para qualquer subconjunto de clientes $S \subseteq V \setminus \{0\}$, a frota deve ser suficiente para atender a demanda.

$$\sum_{i \in S} \sum_{j \in S, j \neq i} x_{ij} \leq |S| - 1$$

 Esta restrição é geralmente adicionada via Callbacks (Cortes).

In [ ]:
def extract_routes_from_x(inst, x_val):
    """Reconstrói rotas a partir do dicionário de arcos ativos x[i,j]=1"""
    adj = defaultdict(list)
    for (i,j) in x_val:
        adj[i].append(j)

    routes = []
    if 0 in adj:
        for start in adj[0]:
            route = [0, start]
            curr = start
            while curr != 0:
                if curr not in adj or not adj[curr]: break # rota quebrada
                nxt = adj[curr][0]
                route.append(nxt)
                curr = nxt
                if curr == 0: break
            routes.append(route)
    return routes

In [ ]:


def solve_branch_and_cut(inst, max_iters=100, time_limit=300):
    print(f"--- B&C Iniciado (CVRP Q={inst.Q}) ---")
    m = pyo.ConcreteModel()
    N = list(range(inst.n + 1))
    A = [(i,j) for i in N for j in N if i != j]

    m.x = pyo.Var(A, within=pyo.Binary)

    # Minimizar distância
    m.obj = pyo.Objective(expr=sum(inst.dist[i,j] * m.x[i,j] for i,j in A))

    m.cons = pyo.ConstraintList()

    # Grau (Assignment Constraints)
    for i in N:
        if i == 0:
            m.cons.add(sum(m.x[0, j] for j in N if j!=0) == inst.K)
            m.cons.add(sum(m.x[j, 0] for j in N if j!=0) == inst.K)
        else:
            m.cons.add(sum(m.x[i, j] for j in N if j!=i) == 1)
            m.cons.add(sum(m.x[j, i] for j in N if j!=i) == 1)

    m.cuts = pyo.ConstraintList()

    _, solver = get_solver(max_seconds=60) # Solver rápido por iteração
    print("Solver: ", _)
    start_time = time.time()

    for it in range(max_iters):
        res = solver.solve(m, tee=False)

        # Recupera arcos
        x_sol = {(i,j) for i,j in A if pyo.value(m.x[i,j]) > 0.5}

        # Cria grafo para análise
        G = nx.DiGraph()
        G.add_nodes_from(N)
        G.add_edges_from(x_sol)

        # Separação
        new_cuts = 0
        violation = False

        # 1. Subciclos desconectados do depósito
        comps = list(nx.strongly_connected_components(G))
        for comp in comps:
            if 0 not in comp and len(comp) >= 2:
                # Subciclo isolado detectado
                violation = True
                S = list(comp)
                # Corte: x(S:S) <= |S| - 1
                expr = sum(m.x[i,j] for i in S for j in S if (i,j) in A)
                m.cuts.add(expr <= len(S) - 1)
                new_cuts += 1

        # 2. Violação de Capacidade em rotas conectadas ao depósito
        # Reconstrói rotas para checar carga
        current_routes = extract_routes_from_x(inst, x_sol)
        for r in current_routes:
            # r é [0, 1, 5, 0] etc.
            # Demanda = len(r) - 2 (desconta os dois zeros)
            load = len(r) - 2
            if load > inst.Q:
                violation = True
                # Rota excede capacidade. Adicionar corte no conjunto de clientes S
                S = [node for node in r if node != 0]

                # Rounded Capacity Inequality
                # x(S:S) <= |S| - ceil(Demand(S)/Q)
                min_vehicles = math.ceil(load / inst.Q)
                expr = sum(m.x[i,j] for i in S for j in S if (i,j) in A)
                m.cuts.add(expr <= len(S) - min_vehicles)
                new_cuts += 1

        if not violation:
            print(f"B&C It {it}: Ótimo Encontrado! Obj={pyo.value(m.obj):.2f}")
            break

        if it % 1 == 0:
            print(f"B&C It {it}: Obj={pyo.value(m.obj):.2f}, Cortes={new_cuts}")

        if time.time() - start_time > time_limit:
            print("Time Limit.")
            break

    total_time = (time.time() - start_time)
    print(f"Tempo de execução Branch-and-Cut: { total_time:.2f}s")

    return pyo.value(m.obj), extract_routes_from_x(inst, {(i,j) for i,j in A if pyo.value(m.x[i,j]) > 0.9})


## Parte II — Branch-and-Price (heurístico de geração de colunas)

Master relaxado com colunas (rotas); duais $(\pi,\mu)$; custo reduzido $\bar c_r = c_r - \sum_i \pi_i a_{ir} - \mu$.  
Sementes longas (k-means + vizinho mais próximo) visam boa factibilidade; fallback sem limite de veículos durante a fase LP.


## Formulação para Branch-and-Price (Set Partitioning)

O Branch-and-Price utiliza a decomposição de Dantzig-Wolfe, onde o problema mestre seleciona rotas prontas.

Modelo Matemático: Branch-and-Price (Set Partitioning)

O problema é decomposto em um Problema Mestre (Master Problem) e um Subproblema (Pricing Problem).

Problema Mestre (Set Partitioning):

Seja $\Omega$ o conjunto de todas as rotas viáveis possíveis.

Variáveis:

$$\lambda_r = \begin{cases} 1 & \text{se a rota } r \in \Omega \text{ é selecionada} \\ 0 & \text{caso contrário} \end{cases}$$

Formulação:

$$\min \sum_{r \in \Omega} c_r \lambda_r$$

Sujeito a:

$$\sum_{r \in \Omega} a_{ir} \lambda_r = 1 \quad \forall i \in V \setminus \{0\}$$

$$\sum_{r \in \Omega} \lambda_r \le K$$

$$\lambda_r \in \{0, 1\}$$

Onde $a_{ir}$ é 1 se a rota $r$ visita o cliente $i$, e $c_r$ é o custo da rota.



### Pricing Problem (Subproblema):

O objetivo é encontrar uma nova rota $r'$ com custo reduzido negativo. Isso geralmente é modelado como um Elementary Shortest Path Problem with Resource Constraints (ESPPRC).

Função Objetivo: Minimizar o Custo Reduzido ($\bar{c}$)


$$\min \bar{c} = \sum_{(i,j) \in A} c_{ij} x_{ij} - \sum_{i \in V \setminus \{0\}} \pi_i \left( \sum_{j \in V} x_{ij} \right) - \sigma$$

Explicação: O custo reduzido é o custo real da rota menos os "prêmios" (duais $\pi_i$) coletados por visitar os clientes, menos o custo fixo do veículo (dual $\sigma$).Sujeito a:Fluxo no Depósito (Fonte e Sumidouro):O veículo deve sair do depósito uma vez e retornar ao depósito uma vez.

$$\sum_{j \in V \setminus \{0\}} x_{0j} = 1$$

$$\sum_{i \in V \setminus \{0\}} x_{i0} = 1$$

Conservação de Fluxo (Clientes):Se o veículo entra em um cliente $k$, ele deve sair de $k$.

$$\sum_{i \in V} x_{ik} - \sum_{j \in V} x_{kj} = 0, \quad \forall k \in V \setminus \{0\}$$

Elementaridade (Visitar no máximo uma vez):Garante que nenhum cliente seja visitado mais de uma vez na mesma rota (ciclos internos são proibidos).

$$\sum_{i \in V} x_{ik} \le 1, \quad \forall k \in V \setminus \{0\}$$

Capacidade e Eliminação de Subciclos (MTZ Constraints):Estas restrições garantem que a carga $u$ aumenta ao visitar nós e que a capacidade $Q$ não é violada. Também impedem subciclos desconectados do depósito.

$$u_j \ge u_i + q_j - M (1 - x_{ij}), \quad \forall (i,j) \in A, j \neq 0$$

$$q_i \le u_i \le Q, \quad \forall i \in V \setminus \{0\}$$

(Onde $M$ é um número grande, geralmente $Q$ é suficiente).
Restrições de Domínio:

$$x_{ij} \in \{0, 1\}$$

$$u_i \ge 0$$

In [ ]:
def run_grasp_vrp(inst, iterations=50, alpha=0.2):
    """
    Gera colunas iniciais usando GRASP (Greedy Randomized Adaptive Search Procedure).
    Baseado em Vizinho Mais Próximo com RCL (Restricted Candidate List).

    Retorna uma lista de dicionários de rotas no formato usado pelo B&P.
    """
    print(f"--- Iniciando Warm-Up GRASP ({iterations} iterações, alpha={alpha}) ---")

    generated_routes = []
    # Usamos um set de tuplas para evitar duplicatas exatas de rotas
    unique_paths = set()

    for it in range(iterations):
        unvisited = set(range(1, inst.n + 1))
        sol_routes = []

        # Tenta construir K veículos (ou mais se necessário, mas tentamos respeitar K)
        # Se sobrar cliente, a solução é inviável, mas as rotas geradas ainda são úteis!

        current_vehicle = 0
        while unvisited:
            current_vehicle += 1
            path = [0]
            curr = 0
            load = 0

            while True:
                # 1. Identificar candidatos viáveis (Capacidade)
                candidates = []
                for node in unvisited:
                    if load + 1 <= inst.Q: # Assumindo demanda 1
                        dist = inst.dist[curr, node]
                        candidates.append((dist, node))

                if not candidates:
                    break # Ninguém cabe ou unvisited vazio

                # 2. Construir RCL (Restricted Candidate List)
                # Ordena pelo mais próximo
                candidates.sort(key=lambda x: x[0])
                min_dist = candidates[0][0]
                threshold = min_dist * (1 + alpha)

                # Pega todos dentro do limiar (alpha)
                rcl = [c for c in candidates if c[0] <= threshold]

                # 3. Escolha Aleatória
                chosen_dist, chosen_node = random.choice(rcl)

                # Atualiza estado
                path.append(chosen_node)
                load += 1
                curr = chosen_node
                unvisited.remove(chosen_node)

            # Fecha rota retornando ao depósito
            path.append(0)

            # Opcional: Aplica 2-opt na rota gerada para garantir qualidade
            # (Já que temos a função pronta, vale a pena usar)
            final_path = improve_route_2opt(path, inst.dist)

            # Converte para formato hashable para checar duplicidade
            path_tuple = tuple(final_path)

            if path_tuple not in unique_paths:
                unique_paths.add(path_tuple)

                # Calcula custo e metadados
                cost = sum(inst.dist[final_path[k], final_path[k+1]] for k in range(len(final_path)-1))
                edges = set((final_path[k], final_path[k+1]) for k in range(len(final_path)-1))

                route_dict = {
                    "path": list(final_path),
                    "cost": cost,
                    "edges": edges,
                    "covered": set(final_path) - {0},
                    "load": len(final_path) - 2 # descontando os dois zeros
                }
                generated_routes.append(route_dict)

    print(f"   -> GRASP gerou {len(generated_routes)} colunas únicas.")
    return generated_routes

In [ ]:
# --- FUNÇÕES AUXILIARES DE PRICING ---

def improve_route_2opt(path, dist_matrix):
    """Aplica 2-opt em um caminho [0, i, ..., 0]"""
    # Trabalha apenas com a parte interna (clientes) para simplificar
    # path entra como [0, 1, 2, 0]
    if len(path) < 4: return path # Sem arestas suficientes para trocar

    best_path = path[:]
    improved = True

    while improved:
        improved = False
        # Índices dos clientes: de 1 a len-2
        for i in range(1, len(best_path) - 2):
            for j in range(i + 1, len(best_path) - 1):
                if j - i == 1: continue

                # Checa ganho de troca
                u, v = best_path[i-1], best_path[i]
                x, y = best_path[j], best_path[j+1]

                current_cost = dist_matrix[u,v] + dist_matrix[x,y]
                new_cost = dist_matrix[u,x] + dist_matrix[v,y]

                if new_cost < current_cost - 1e-4:
                    # Inverte segmento i..j
                    best_path[i:j+1] = best_path[i:j+1][::-1]
                    improved = True
    return best_path

In [ ]:
def solve_exact_pricing(inst, pi_cover, pi_fleet, forbidden_arcs, forced_arcs, Q):
    """MIP para encontrar Caminho Elementar Mais Curto com Recurso (Capacidade)"""
    m = pyo.ConcreteModel()
    nodes = list(range(inst.n + 1))

    # --- LÓGICA DE FILTRAGEM DE ARCOS ---
    # Precisamos criar uma lista de arcos que respeite tanto Forbidden quanto Forced
    valid_arcs = []

    # Pré-processamento dos forçados para acesso rápido
    # forced_out[u] = v  significa que u TEM que ir para v
    forced_out = {}
    forced_in = {}
    for (u, v) in forced_arcs:
        forced_out[u] = v
        forced_in[v] = u

    for i in nodes:
        for j in nodes:
            if i == j: continue

            # 1. Checa Forbidden
            if (i, j) in forbidden_arcs:
                continue

            # 2. Checa Forced OUT: Se i tem que ir para X, e j != X, então (i,j) é inválido
            if i in forced_out and forced_out[i] != j:
                continue

            # 3. Checa Forced IN: Se j tem que vir de Y, e i != Y, então (i,j) é inválido
            if j in forced_in and forced_in[j] != i:
                continue

            valid_arcs.append((i,j))

    m.x = pyo.Var(valid_arcs, within=pyo.Binary)
    m.u = pyo.Var(nodes, within=pyo.NonNegativeReals, bounds=(0, Q))

    # Custo Reduzido
    def red_cost_rule(model):
        real = sum(inst.dist[i,j] * m.x[i,j] for (i,j) in valid_arcs)
        gains = sum(pi_cover[i] * sum(m.x[i,j] for j in nodes if (i,j) in valid_arcs)
                    for i in range(1, inst.n+1))
        return real - gains - pi_fleet

    m.obj = pyo.Objective(rule=red_cost_rule, sense=pyo.minimize)

    m.cons = pyo.ConstraintList()

    # Fluxo
    m.cons.add(sum(m.x[0,j] for j in nodes if (0,j) in valid_arcs) == 1)
    m.cons.add(sum(m.x[i,0] for i in nodes if (i,0) in valid_arcs) == 1)

    # Conservação
    for k in range(1, inst.n+1):
        incoming = sum(m.x[i,k] for i in nodes if (i,k) in valid_arcs)
        outgoing = sum(m.x[k,j] for j in nodes if (k,j) in valid_arcs)
        m.cons.add(incoming == outgoing)
        m.cons.add(incoming <= 1)

    # MTZ / Capacidade
    for (i,j) in valid_arcs:
        if j != 0:
            m.cons.add(m.u[j] >= m.u[i] + 1 - Q*(1 - m.x[i,j]))

    # Solver com Time Limit um pouco maior para garantir gap fechado no pricing
    _, solver = get_solver(max_seconds=300)
    if not solver: return None, 0.0

    solver.solve(m, tee=False)

    min_rc_val = pyo.value(m.obj)

    # Tolerância numérica segura
    if min_rc_val < -1e-5:
        path = [0]
        curr = 0
        visited_edges = set() # Segurança contra loops infinitos na reconstrução
        while True:
            nxt = None
            for j in nodes:
                if (curr,j) in valid_arcs and pyo.value(m.x[curr,j]) > 0.5:
                    nxt = j; break
            if nxt is None: break

            if (curr, nxt) in visited_edges: break # Loop detectado
            visited_edges.add((curr, nxt))

            path.append(nxt)
            curr = nxt
            if curr == 0: break

        real_cost = sum(inst.dist[path[k], path[k+1]] for k in range(len(path)-1))
        edges = set((path[k], path[k+1]) for k in range(len(path)-1))

        col = {
            "path": path,
            "cost": real_cost,
            "edges": edges,
            "covered": set(path) - {0},
            "load": len(path) - 2
        }
        return col, min_rc_val

    return None, min_rc_val


In [ ]:
# --- CLASSE E ALGORITMO B&P ---

class BPNode:
    def __init__(self, nid, parent, forbidden, forced):
        self.id = nid
        self.forbidden = set(forbidden)
        self.forced = set(forced)
        self.lb = -float('inf')
        self.x_sol = {} # Solução fracionária nos arcos

    def __lt__(self, other):
        return self.lb < other.lb


In [ ]:
def solve_node(node, global_routes, alpha=0.5, gap_tol=0.005, time_limit=900, max_cg_iter=10000):
    # 1. Ajusta matriz de custo para proibir arcos no Pricing Heurístico
    dist_h = inst.dist.copy()
    BIG = 1e6
    for (u,v) in node.forbidden: dist_h[u,v] = BIG
    for (u,v) in node.forced:
        # Se forçado u->v, proibe u->k e k->v
        for k in range(inst.n+1):
            if k!=v: dist_h[u,k] = BIG
            if k!=u: dist_h[k,v] = BIG

    loop_cols = True
    iter_cg = 0

    # --- Variáveis para Estabilização ---
    pi_cov_stab = {k: 0.0 for k in range(1, inst.n+1)}
    pi_flt_stab = 0.0
    first_iter = True

    best_lb = 0
    best_gap = float('inf')

    while loop_cols and iter_cg < max_cg_iter:
        iter_cg += 1
        # -- RMP --
        rmp = pyo.ConcreteModel()

        # Filtra rotas válidas (Branching)
        valid_idx = []
        for idx, r in enumerate(global_routes):
            if r["edges"].isdisjoint(node.forbidden):
                # Checa forced
                ok = True
                for (u,v) in node.forced:
                    # Se tem u, deve ser seguido por v
                    p = r["path"]
                    if u in p[:-1]:
                        if p[p.index(u)+1] != v: ok = False; break
                    if v in p[1:]:
                        if p[p.index(v)-1] != u: ok = False; break
                if ok: valid_idx.append(idx)

        rmp.lam = pyo.Var(valid_idx, within=pyo.NonNegativeReals)
        rmp.slk_cov = pyo.Var(range(1, inst.n+1), within=pyo.NonNegativeReals) # Slack Cobertura
        rmp.slk_flt = pyo.Var(within=pyo.NonNegativeReals) # Slack Frota

        M_PENALTY = 100000.0

        rmp.obj = pyo.Objective(expr=
            sum(global_routes[i]["cost"] * rmp.lam[i] for i in valid_idx) +
            sum(M_PENALTY * rmp.slk_cov[k] for k in range(1, inst.n+1)) +
            M_PENALTY * rmp.slk_flt
        )

        rmp.c_cov = pyo.ConstraintList()
        for k in range(1, inst.n+1):
            rmp.c_cov.add(sum(rmp.lam[i] for i in valid_idx if k in global_routes[i]["covered"]) + rmp.slk_cov[k] == 1)

        rmp.c_flt = pyo.Constraint(expr=sum(rmp.lam[i] for i in valid_idx) + rmp.slk_flt == inst.K)

        rmp.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

        _, s = get_solver(60)

        if not s: return False
        res = s.solve(rmp, tee=False)

        if res.solver.termination_condition != pyo.TerminationCondition.optimal:
            node.lb = float('inf'); return True

        current_obj = pyo.value(rmp.obj)

        print(f"Node: {node.id} / Iter: {iter_cg} / LB: {best_lb:.2f} / UB: {current_obj:.2f} / gap: {best_gap:.2f} / Solver: {_}")

        # 2. Recupera Duais Reais
        try:
            pi_cov_curr = {k: rmp.dual[rmp.c_cov[k]] for k in range(1, inst.n+1)}
            pi_flt_curr = rmp.dual[rmp.c_flt]
        except:
            node.lb = float('inf'); return True


        # --- Estabilização (Smoothing) ---
        # pi_stab = alpha * pi_curr + (1-alpha) * pi_prev
        if first_iter:
            pi_cov_stab = pi_cov_curr.copy()
            pi_flt_stab = pi_flt_curr
            first_iter = False
        else:
            for k in range(1, inst.n+1):
                pi_cov_stab[k] = alpha * pi_cov_curr[k] + (1 - alpha) * pi_cov_stab[k]
            pi_flt_stab = alpha * pi_flt_curr + (1 - alpha) * pi_flt_stab

        # Para o pricing, usamos os duais ESTABILIZADOS
        pi_cov_use = pi_cov_stab
        pi_flt_use = pi_flt_stab

        # --- Pricing Heurístico (Usa duais estabilizados) ---
        new_col = False
        seeds = list(range(1, inst.n+1)); random.shuffle(seeds)
        for start in seeds:
            path = [0, start]; load = 1; cost = dist_h[0, start]
            if load > inst.Q or cost > BIG/2: continue
            curr = start; vis = {start}
            while load < inst.Q:
                close_c = dist_h[curr, 0]
                if close_c < BIG/2:
                    rc_path = (cost + close_c) - sum(pi_cov_use[x] for x in vis) - pi_flt_use
                    if rc_path < -1e-2: # Tolerância heurística
                        raw_p = path + [0]
                        # Usamos dist_h em vez de inst.dist.
                        # Isso garante que o 2-opt não reintroduza arcos proibidos (custo BIG).
                        # Matematicamente, minimizar dist_h é igual a minimizar Custo Reduzido aqui.
                        opt_p = improve_route_2opt(raw_p, dist_h)
                        opt_p = raw_p
                        edges_opt = set((opt_p[k], opt_p[k+1]) for k in range(len(opt_p)-1))
                        if edges_opt.isdisjoint(node.forbidden):
                            c_opt = sum(inst.dist[opt_p[k], opt_p[k+1]] for k in range(len(opt_p)-1))
                            global_routes.append({"path": opt_p, "cost": c_opt, "edges": edges_opt, "covered": set(opt_p)-{0}, "load": load})
                            #print(f"    Coluna heurística: {opt_p}, valor: {c_opt}, cr: {rc_path:.2f}, obj: {current_obj:.2f}",)
                            new_col = True
                bst_n = None; bst_val = float('inf')
                for nxt in range(1, inst.n+1):
                    if nxt not in vis and dist_h[curr, nxt] < BIG/2:
                        val = dist_h[curr, nxt] - pi_cov_use[nxt]
                        if val < bst_val: bst_val = val; bst_n = nxt
                if bst_n is not None and (bst_val < 1.0 or len(path) < 3):
                    path.append(bst_n); vis.add(bst_n); curr = bst_n
                    cost += dist_h[path[-2], bst_n]; load += 1
                else: break

        # -- PRICING EXATO (FALLBACK) --
        if not new_col:
            ex_col, min_rc = solve_exact_pricing(inst, pi_cov_curr, pi_flt_curr,
                                                  node.forbidden, node.forced, inst.Q)
            if min_rc > -1e-5:
                loop_cols = False
                lagrangian_lb = current_obj
                break

            # --- Cálculo do Lagrangian Bound ---
            # LB = z_RMP + K * min_reduced_cost
            lagrangian_lb = current_obj + inst.K * min_rc
            best_lb = max(best_lb, lagrangian_lb)

            # Gap relativo
            gap = 0.0
            if current_obj > 1e-6:
                gap = (current_obj - lagrangian_lb) / current_obj
            best_gap = min(best_gap, gap)

            # Debug (Opcional)
            # if iter_cg % 5 == 0:
            #    print(f"   Iter {iter_cg}: UB={current_obj:.2f} LB={lagrangian_lb:.2f} Gap={gap*100:.2f}%")

            if ex_col:
                global_routes.append(ex_col)
                #print(f"    !!! Coluna exata: {ex_col['path']}, valor: {ex_col['cost']:.2f}, , cr: {min_rc:.2f}, obj: {current_obj:.2f}, lb: {lagrangian_lb:.2f}",)
                new_col = True

            # Critério de Parada por Bound ---
            # Se o gap for muito pequeno (ex: 0.5%), paramos mesmo que ainda tenha coluna negativa
            if gap < gap_tol and current_obj < 10000: # < 10000 garante que é factível
                # print(f"   -> Parada antecipada! Gap {gap*100:.2f}% < {gap_tol*100}%")
                loop_cols = False


        loop_cols = new_col


        # Salva estado se parou
        if not loop_cols:
            node.lb = pyo.value(rmp.obj)
            # Verifica penalidade
            pen = sum(pyo.value(rmp.slk_cov[k]) for k in range(1,inst.n+1)) + pyo.value(rmp.slk_flt)
            if pen > 1e-3: node.lb = float('inf')

            # Salva x_ij para branching
            node.x_sol = defaultdict(float)
            for i in valid_idx:
                val = pyo.value(rmp.lam[i])
                if val > 1e-4:
                    p = global_routes[i]["path"]
                    for k in range(len(p)-1):
                        node.x_sol[(p[k], p[k+1])] += val


In [ ]:
# --- BRANCH-AND-PRICE ---

def solve_full_branch_and_price(inst, alpha=0.5, gap_tol=0.005, time_limit=900, max_cg_iter=10000):
    print(f"--- B&P Iniciado (CVRP Q={inst.Q}) ---")
    start_time = time.time()

    # Pool Global
    global_routes = []
    # Init com pendulares
    for i in range(1, inst.n+1):
        c = inst.dist[0,i] + inst.dist[i,0]
        global_routes.append({
            "path": [0, i, 0],
            "cost": c,
            "edges": {(0,i), (i,0)},
            "covered": {i},
            "load": 1
        })


    # --- NOVO: WARM-UP COM GRASP ---
    # Gera 100 iterações de GRASP com alpha=0.2 (20% de aleatoriedade no guloso)
    grasp_cols = run_grasp_vrp(inst, iterations=100, alpha=0.2)

    # Adiciona ao pool global
    global_routes.extend(grasp_cols)

    # -- ARVORE B&B --
    root = BPNode(0, -1, [], [])
    pq = []
    solve_node(root, global_routes, alpha, gap_tol, time_limit, max_cg_iter)
    heappush(pq, root)

    best_int_val = float('inf')
    best_int_routes = []
    cnt = 0

    while pq:
        if time.time() - start_time > time_limit: break
        node = heappop(pq)

        if node.lb >= best_int_val or node.lb == float('inf'): continue

        print(f"Node {node.id}: LB={node.lb:.2f}")

        # Branching
        frac_arc = None
        closest = 0.5
        for (u,v), val in node.x_sol.items():
            if abs(val - round(val)) > 1e-3:
                if abs(val - 0.5) < closest:
                    closest = abs(val - 0.5); frac_arc = (u,v)

        if frac_arc is None:
            print(f"  -> Inteiro! {node.lb:.2f}")
            if node.lb < best_int_val:
                best_int_val = node.lb
                # Recupera rotas (Simplificado, idealmente pega do RMP do nó)
                # Aqui vamos confiar que o último RMP Inteiro Final resolverá
            continue

        # Ramifica
        u, v = frac_arc
        print(f"  -> Branching ({u},{v})")
        cnt += 1
        c0 = BPNode(cnt, node.id, node.forbidden | {(u,v)}, node.forced)
        solve_node(c0);
        if c0.lb < best_int_val: heappush(pq, c0)

        cnt += 1
        c1 = BPNode(cnt, node.id, node.forbidden, node.forced | {(u,v)})
        solve_node(c1);
        if c1.lb < best_int_val: heappush(pq, c1)

    # MESTRE FINAL INTEIRO
    print("\n--- IP Final ---")
    rmp_f = pyo.ConcreteModel()
    rmp_f.lam = pyo.Var(range(len(global_routes)), within=pyo.Binary)
    rmp_f.slk = pyo.Var(within=pyo.NonNegativeReals)

    rmp_f.obj = pyo.Objective(expr=sum(global_routes[i]["cost"]*rmp_f.lam[i] for i in range(len(global_routes))) + 1e5*rmp_f.slk)
    rmp_f.cov = pyo.ConstraintList()
    for k in range(1, inst.n+1):
        rmp_f.cov.add(sum(rmp_f.lam[i] for i in range(len(global_routes)) if k in global_routes[i]["covered"]) == 1)
    rmp_f.flt = pyo.Constraint(expr=sum(rmp_f.lam[i] for i in range(len(global_routes))) + rmp_f.slk == inst.K)

    _, s = get_solver(600)
    s.solve(rmp_f)

    final_routes = []
    if pyo.value(rmp_f.obj) < 1e5:
        for i in range(len(global_routes)):
            if pyo.value(rmp_f.lam[i]) > 0.5: final_routes.append(global_routes[i]["path"])
    total_time = (time.time() - start_time)
    print(f"Tempo de execução Branch-and-Price: {total_time:.2f}s")

    return pyo.value(rmp_f.obj), final_routes

In [ ]:
# --- CRIAÇÃO DA INSTÂNCIA ---
inst = make_instance(n=12, K=3, Q=7, seed=37)
plot_routes(inst, [], "Instância VRP ")

In [ ]:
# EXECUÇÃO COMPARATIVA
print("=== PARTE 1: BRANCH-AND-CUT ===")
cost_bc, routes_bc = solve_branch_and_cut(inst, max_iters=50)
plot_routes(inst, routes_bc, f"Branch-and-Cut (Q={inst.Q}) - Custo {cost_bc:.2f}")

In [ ]:
print("\n=== PARTE 2: BRANCH-AND-PRICE ===")
cost_bp, routes_bp = solve_full_branch_and_price(inst, alpha=0.5, gap_tol=0.01, time_limit=3600)
plot_routes(inst, routes_bp, f"Branch-and-Price (Q={inst.Q}) - Custo {cost_bp:.2f}")

In [ ]:
def solve_full_branch_and_cut(inst, time_limit=300):
    print(f"--- Branch-and-Cut Manual ROBUSTO (CVRP Q={inst.Q}) ---")
    start_time = time.time()

    N = list(range(inst.n + 1))
    A = [(i,j) for i in N for j in N if i != j]

    # --- Função auxiliar para extrair rotas de solução inteira ---
    def extract_routes_check(x_sol):
        """Reconstroi rotas e verifica capacidade. Retorna (Rotas, is_valid, violated_sets)"""
        # Constrói grafo de adjacência
        adj = defaultdict(list)
        for (i,j), val in x_sol.items():
            if val > 0.5:
                adj[i].append(j)

        routes = []
        violated_sets = []
        is_feasible = True

        # Reconstrói rotas a partir do 0
        try:
            # O 0 pode ter múltiplas saídas
            starts = adj[0]
            visited_global = {0}

            for start_node in starts:
                curr = start_node
                route = [0, curr]
                visited_global.add(curr)

                # Segue o caminho até voltar a 0
                while curr != 0:
                    if not adj[curr]: break # Rota quebrada (não deveria acontecer se grau=1)
                    nxt = adj[curr][0] # Assume integridade binária
                    route.append(nxt)
                    if nxt != 0: visited_global.add(nxt)
                    curr = nxt

                    if len(route) > inst.n + 2: break # Segurança loop infinito

                # Valida Capacidade
                clients = [c for c in route if c != 0]
                load = len(clients) # Demanda unitária

                if load > inst.Q:
                    is_feasible = False
                    violated_sets.append(set(clients))

                routes.append(route)

        except Exception as e:
            is_feasible = False # Erro na reconstrução

        return routes, is_feasible, violated_sets

    # ... (solve_lp_node permanece IDÊNTICA à anterior) ...
    def solve_lp_node(node):
        m = pyo.ConcreteModel()
        m.x = pyo.Var(A, within=pyo.NonNegativeReals, bounds=(0,1))
        m.obj = pyo.Objective(expr=sum(inst.dist[i,j] * m.x[i,j] for i,j in A))
        m.cons = pyo.ConstraintList()

        # Grau
        for i in N:
            if i == 0:
                m.cons.add(sum(m.x[0, j] for j in N if j!=0) == inst.K)
                m.cons.add(sum(m.x[j, 0] for j in N if j!=0) == inst.K)
            else:
                m.cons.add(sum(m.x[i, j] for j in N if j!=i) == 1)
                m.cons.add(sum(m.x[j, i] for j in N if j!=i) == 1)

        # Branching
        for (u,v) in node.fixed_0: m.x[u,v].fix(0)
        for (u,v) in node.fixed_1: m.x[u,v].fix(1)

        # Cortes
        for c_type, S, rhs in node.cuts:
            m.cons.add(sum(m.x[i,j] for i in S for j in S if (i,j) in A) <= rhs)

        cut_loop_iter = 0
        max_cut_loop = 15 # Aumentei um pouco

        while True:
            _, solver = get_solver(max_seconds=5) # Rápido
            if not solver: return False
            res = solver.solve(m, tee=False)

            if res.solver.termination_condition != pyo.TerminationCondition.optimal:
                node.lb = float('inf'); return True

            current_x = {(i,j): pyo.value(m.x[i,j]) for i,j in A}
            node.lb = pyo.value(m.obj)
            node.x_sol = current_x # Salva sempre a última viável

            if cut_loop_iter >= max_cut_loop: break

            new_violated = find_violated_cuts(inst, current_x)
            if not new_violated: break

            added_any = False
            for c_type, S, rhs in new_violated:
                # Checagem simples para não explodir o LP com cortes repetidos
                # (Idealmente usaria um set de hashes dos cortes)
                m.cons.add(sum(m.x[i,j] for i in S for j in S if (i,j) in A) <= rhs)
                node.cuts.append((c_type, S, rhs))
                added_any = True

            if not added_any: break
            cut_loop_iter += 1

        return True

    # --- MAIN LOOP ---
    root = BCNode(0, -1, [], [], [])
    pq = []

    print("Processando Raiz...")
    solve_lp_node(root)
    heappush(pq, root)

    best_int_val = float('inf')
    best_int_routes = []
    cnt = 0

    while pq:
        if time.time() - start_time > time_limit: break
        node = heappop(pq)

        if node.lb >= best_int_val or node.lb == float('inf'): continue

        print(f"Node {node.id}: LB={node.lb:.2f} | Cortes={len(node.cuts)}")

        # Checa integridade
        frac_arc = None
        closest = 0.5
        for (u,v), val in node.x_sol.items():
            if abs(val - round(val)) > 1e-3:
                dist = abs(val - 0.5)
                if dist < closest: closest = dist; frac_arc = (u,v)

        if frac_arc is None:
            # --- CORREÇÃO CRÍTICA AQUI ---
            # Solução é Inteira. Mas é Viável na Capacidade?

            x_int = {k: v for k,v in node.x_sol.items() if v > 0.9}
            routes, is_feasible, violated_sets = extract_routes_check(x_int)

            if is_feasible:
                print(f"  -> Solução Inteira VIÁVEL! Custo: {node.lb:.2f}")
                if node.lb < best_int_val:
                    best_int_val = node.lb
                    best_int_routes = routes
            else:
                print(f"  -> Solução Inteira INVÁVEL (Capacidade). Adicionando cortes...")
                # Adiciona cortes para os conjuntos violados e RE-RESOLVE este nó
                # Em vez de ramificar, nós simplesmente fortalecemos este nó e o jogamos de volta na fila
                # ou resolvemos agora mesmo. Vamos re-resolver agora.

                new_cuts_added = False
                for S_viol in violated_sets:
                    s_list = list(S_viol)
                    demand = len(s_list)
                    min_k = math.ceil(demand / inst.Q)
                    rhs = len(s_list) - min_k

                    # Verifica se corte já existe (simples)
                    # Adiciona corte GSE Exato para este conjunto
                    node.cuts.append(('GSE', s_list, rhs))
                    new_cuts_added = True

                if new_cuts_added:
                    # Re-resolve o LP deste nó com os novos cortes
                    solve_lp_node(node)
                    # Se ainda promissor, devolve para a fila
                    if node.lb < best_int_val:
                        heappush(pq, node)

            continue # Pula branching (já tratamos ou era ótima)

        # Branching (Padrão)
        u, v = frac_arc
        # print(f"  -> Branching ({u},{v})")
        cnt += 1
        c0 = BCNode(cnt, node.id, node.fixed_0 | {(u,v)}, node.fixed_1, list(node.cuts))
        solve_lp_node(c0)
        if c0.lb < best_int_val: heappush(pq, c0)

        cnt += 1
        c1 = BCNode(cnt, node.id, node.fixed_0, node.fixed_1 | {(u,v)}, list(node.cuts))
        solve_lp_node(c1)
        if c1.lb < best_int_val: heappush(pq, c1)

    total_time = (time.time() - start_time)
    print(f"Tempo de execução Branch-and-Cut: {total_time:.2f}s")
    return best_int_val, best_int_routes

In [ ]:
# Execução
cost_bc_manual, routes_bc_manual = solve_full_branch_and_cut(inst, time_limit=300)
plot_routes(inst, routes_bc_manual, f"B&C Manual (Q={inst.Q}) - Custo {cost_bc_manual:.2f}")

In [ ]:
print(f"\nResumo Final:\nB&C: {cost_bc:.2f}\nB&P: {cost_bp:.2f}")